# Get Embedding from STATE  

## 1. Setup and Patches

Apply necessary patches to torch.load and Lightning for compatibility.

In [1]:
import sys
import os
import torch

# Patch torch.load to always set weights_only=False
# optionally, you can remove this patch if not needed
_original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_load(*args, **kwargs)
torch.load = patched_load

print("✓ Torch.load patched")

✓ Torch.load patched


In [2]:
# Add DictConfig to safe globals for torch serialization
# optionally, you can remove this patch if not needed
import torch.serialization
from omegaconf.dictconfig import DictConfig
torch.serialization.add_safe_globals([DictConfig])

print("✓ Safe globals added")

✓ Safe globals added


In [3]:
# Patch Lightning Fabric cloud_io._load to always set weights_only=False
# optionally, you can remove this patch if not needed
import lightning.fabric.utilities.cloud_io as cloud_io
_original_pl_load = cloud_io._load
def patched_pl_load(path_or_url, map_location=None, weights_only=None):
    return _original_pl_load(path_or_url, map_location=map_location, weights_only=False)
cloud_io._load = patched_pl_load

print("✓ Lightning patched")

✓ Lightning patched


## 2. Import Required Modules

In [5]:
from pathlib import Path
from omegaconf import OmegaConf
from src.state.emb.inference import Inference   # adjusted import path from https://github.com/ArcInstitute/state

print("✓ All modules imported")

✓ All modules imported


## 3. Configuration Parameters

Set your input/output paths and parameters here.

In [6]:
# Configuration
# model files are from https://huggingface.co/arcinstitute/SE-600M/tree/main
model_folder = "./modelfile"
checkpoint = "./modelfile/se600m_epoch15.ckpt"
input_file = "./LINCS_test500pairs.h5ad"
output_file = "./output/LINCS_test500pairs_wEmmbedding.h5ad"
dataset_name = "perturbation"
embed_key = "X_emb"
gene_column = "gene_name"

print("Configuration set:")
print(f"  Model folder: {model_folder}")
print(f"  Checkpoint: {checkpoint}")
print(f"  Input: {input_file}")
print(f"  Output: {output_file}")

Configuration set:
  Model folder: ./modelfile
  Checkpoint: ./modelfile/se600m_epoch15.ckpt
  Input: ./LINCS_test500pairs.h5ad
  Output: ./output/LINCS_test500pairs_wEmmbedding.h5ad


## 4. Helper Function: Load Config

In [7]:
def load_config_from_model_folder(model_folder):
    """Load config and find embeddings file"""
    model_folder = Path(model_folder)
    
    possible_configs = [
        model_folder / "config.yaml",
        model_folder / "training_config.yaml",
        model_folder / "hparams.yaml",
    ]
    
    conf = None
    for config_path in possible_configs:
        if config_path.exists():
            print(f"Loading config from: {config_path}")
            conf = OmegaConf.load(config_path)
            break
    
    if conf is None:
        conf = OmegaConf.create({})
    
    # Find embeddings file
    embedding_candidates = [
        'protein_embeddings.pt',  # from https://huggingface.co/arcinstitute/SE-600M/tree/main
        'Homo_sapiens.GRCh38.gene_symbol_to_embedding_ESM2.pt',
        'embeddings.pt',
    ]
    
    embedding_file = None
    for candidate in embedding_candidates:
        candidate_path = model_folder / candidate
        if candidate_path.exists():
            embedding_file = str(candidate_path)
            print(f"Found embeddings file: {candidate}")
            break
    
    if embedding_file is None:
        raise FileNotFoundError(f"No embeddings file found in {model_folder}")
    
    if 'embeddings' not in conf:
        conf['embeddings'] = OmegaConf.create({})
    if 'current' not in conf.embeddings:
        conf.embeddings['current'] = 'esm2'
    
    current_type = conf.embeddings.current
    if current_type not in conf.embeddings:
        conf.embeddings[current_type] = OmegaConf.create({})
    
    conf.embeddings[current_type]['all_embeddings'] = embedding_file
    
    return conf, embedding_file

print("✓ Helper function defined")

✓ Helper function defined


## 5. Load Configuration and Embeddings

In [8]:
# Load config and embeddings
conf, embedding_file = load_config_from_model_folder(model_folder)

print(f"\nLoading protein embeddings")
protein_embeds = torch.load(embedding_file, weights_only=False)
print(f"✓ Loaded {len(protein_embeds)} protein embeddings")

Loading config from: modelfile/config.yaml
Found embeddings file: protein_embeddings.pt

Loading protein embeddings
✓ Loaded 19790 protein embeddings


## 6. Initialize Inference Model

In [9]:
# Initialize inference with protein_embeds
inferer = Inference(conf, protein_embeds=protein_embeds)

print("✓ Inference object initialized")

✓ Inference object initialized


## 7. Load Model Checkpoint

In [10]:
print("Loading model checkpoint...")
inferer.load_model(checkpoint)
print("✓ Model checkpoint loaded")

Loading model checkpoint...
✓ Model checkpoint loaded


## 8. Create Output Directory

In [11]:
os.makedirs(os.path.dirname(output_file), exist_ok=True)
print(f"✓ Output directory ready: {os.path.dirname(output_file)}")

✓ Output directory ready: ./output


## 9. Run Inference

Compute embeddings for the input data.

In [12]:
print("Running inference...")
inferer.encode_adata(
    input_file, 
    output_file, 
    emb_key=embed_key, 
    dataset_name=dataset_name,
    gene_column=gene_column
)

print(f"\n✓ Embedding complete! Output saved to: {output_file}")

Running inference...
!!! 968 genes mapped to embedding file (out of 978)


Encoding: 100%|██████████| 21/21 [00:20<00:00,  1.03it/s]



✓ Embedding complete! Output saved to: ./output/LINCS_test500pairs_wEmmbedding.h5ad


## 10. Verify Output 

Load and inspect the output file to verify the embeddings were created successfully.

In [13]:
# Load and verify the output
import scanpy as sc

adata_output = sc.read_h5ad(output_file)
print(f"Output AnnData shape: {adata_output.shape}")
print(f"Embeddings stored in: {embed_key}")
if embed_key in adata_output.obsm:
    print(f"Embedding shape: {adata_output.obsm[embed_key].shape}")
    print("✓ Embeddings verified!")

Output AnnData shape: (1000, 978)
Embeddings stored in: X_emb
Embedding shape: (1000, 2058)
✓ Embeddings verified!
